In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# Helper for keeping the latest version of each record
# ---------------------------------------------------------

def latest_record(df, business_key):
    window = (
        Window
        .partitionBy(business_key)
        .orderBy(
            F.col("updated_at").desc(),
            F.col("_airbyte_extracted_at").desc(),
        )
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


# ---------------------------------------------------------
# 1. Campaigns
# ---------------------------------------------------------

bronze_campaigns = spark.table(
    "voyage.bronze.marketing_campaigns"
)

campaigns = (
    bronze_campaigns
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
    .withColumn(
        "start_date",
        F.to_date("start_date"),
    )
    .withColumn(
        "end_date",
        F.to_date("end_date"),
    )
)

silver_campaigns = latest_record(
    campaigns,
    "campaign_id",
).select(
    "campaign_id",
    "tenant_id",
    "campaign_name",
    "channel",
    "campaign_type",
    "objective",
    "status",
    "start_date",
    "end_date",
    "created_at",
    "updated_at",
)

(
    silver_campaigns.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_campaign")
)


# ---------------------------------------------------------
# 2. Ad groups
# ---------------------------------------------------------

bronze_ad_groups = spark.table(
    "voyage.bronze.marketing_ad_groups"
)

ad_groups = (
    bronze_ad_groups
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
)

silver_ad_groups = latest_record(
    ad_groups,
    "ad_group_id",
).select(
    "ad_group_id",
    "campaign_id",
    "tenant_id",
    "ad_group_name",
    "status",
    "created_at",
    "updated_at",
)

(
    silver_ad_groups.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_ad_group")
)


# ---------------------------------------------------------
# 3. Ads
# ---------------------------------------------------------

bronze_ads = spark.table(
    "voyage.bronze.marketing_ads"
)

ads = (
    bronze_ads
    .withColumn(
        "created_at",
        F.to_timestamp("created_at"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
)

silver_ads = latest_record(
    ads,
    "ad_id",
).select(
    "ad_id",
    "ad_group_id",
    "campaign_id",
    "tenant_id",
    "ad_name",
    "creative_type",
    "status",
    "created_at",
    "updated_at",
)

(
    silver_ads.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.dim_ad")
)


# ---------------------------------------------------------
# 4. Campaign daily performance
# ---------------------------------------------------------

bronze_performance = spark.table(
    "voyage.bronze.marketing_campaign_performance"
)

performance = (
    bronze_performance
    .withColumn(
        "date",
        F.to_date("date"),
    )
    .withColumn(
        "updated_at",
        F.to_timestamp("updated_at"),
    )
    .withColumn(
        "impressions",
        F.col("impressions").cast("bigint"),
    )
    .withColumn(
        "clicks",
        F.col("clicks").cast("bigint"),
    )
    .withColumn(
        "conversions",
        F.col("conversions").cast("bigint"),
    )
    .withColumn(
        "spend",
        F.col("spend").cast("decimal(14,2)"),
    )
    .withColumn(
        "revenue",
        F.col("revenue").cast("decimal(14,2)"),
    )
)

silver_performance = latest_record(
    performance,
    "metric_id",
).select(
    "metric_id",
    "date",
    "tenant_id",
    "campaign_id",
    "impressions",
    "clicks",
    "spend",
    "conversions",
    "revenue",
    "updated_at",
)

(
    silver_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_campaign_daily")
)


print("Marketing Silver tables created successfully.")

#### Silver tables validation.

In [0]:
tables = [
    "voyage.silver.dim_campaign",
    "voyage.silver.dim_ad_group",
    "voyage.silver.dim_ad",
    "voyage.silver.fact_campaign_daily",
]

for table in tables:
    print(f"{table}: {spark.table(table).count()} rows")

#### checking of updated metric.

In [0]:
display(
    spark.sql("""
        SELECT
            metric_id,
            campaign_id,
            date,
            spend,
            updated_at
        FROM voyage.silver.fact_campaign_daily
        WHERE metric_id = 1
    """)
)

#### relationship between tables.

In [0]:
print(
    "Orphan ad groups:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.dim_ad_group ag
        LEFT JOIN voyage.silver.dim_campaign c
            ON ag.campaign_id = c.campaign_id
        WHERE c.campaign_id IS NULL
    """).first()[0]
)

print(
    "Orphan ads:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.dim_ad a
        LEFT JOIN voyage.silver.dim_ad_group ag
            ON a.ad_group_id = ag.ad_group_id
        WHERE ag.ad_group_id IS NULL
    """).first()[0]
)

print(
    "Orphan performance:",
    spark.sql("""
        SELECT COUNT(*)
        FROM voyage.silver.fact_campaign_daily p
        LEFT JOIN voyage.silver.dim_campaign c
            ON p.campaign_id = c.campaign_id
        WHERE c.campaign_id IS NULL
    """).first()[0]
)

#### final data mart after DBT transfromation.

In [0]:
%sql
SELECT * FROM voyage.dbt_dev.fct_campaign_daily LIMIT 20;

In [0]:
%sql
select
    campaign_id,                    
    date,                 
    impressions,                  
    clicks,                         
    spend,                           
    conversions,                    
    revenue,
    ctr,                            
    cpc,                    
    conversion_rate,     
    roas
from voyage.dbt_dev.fct_campaign_daily
order by campaign_id, date;

In [0]:
%sql
select
    campaign_id,                    
    date,                 
    revenue,                      
    ctr,                            
    cpc,                             
    conversion_rate,                
    roas
from voyage.dbt_dev.fct_campaign_daily
where campaign_id = 'CMP001'
  and date = '2026-04-01';

In [0]:
%sql
select
    sum(impressions) as impressions,
    sum(clicks) as clicks,
    round(sum(spend), 2) as spend,
    sum(conversions) as conversions,
    round(sum(revenue), 2) as revenue
from voyage.dbt_dev.fct_campaign_daily;

In [0]:
%sql
with silver as (   
    select                         
        count(*) as row_count,
        sum(impressions) as impressions,
        sum(clicks) as clicks,
        round(sum(spend), 2) as spend,
        sum(conversions) as conversions,
        round(sum(revenue), 2) as revenue
    from voyage.silver.fact_campaign_daily
),                   
mart as (            
    select              
        count(*) as row_count,
        sum(impressions) as impressions,
        sum(clicks) as clicks,
        round(sum(spend), 2) as spend,
        sum(conversions) as conversions,
        round(sum(revenue), 2) as revenue
    from voyage.dbt_dev.fct_campaign_daily
)
select
    case
        when s.row_count = m.row_count
         and s.impressions = m.impressions
         and s.clicks = m.clicks
         and s.spend = m.spend
         and s.conversions = m.conversions
         and s.revenue = m.revenue
        then 'PASS'
        else 'FAIL'
    end as reconciliation_status,

    s.row_count as silver_rows,
    m.row_count as mart_rows,
    s.impressions - m.impressions as impressions_diff,
    s.clicks - m.clicks as clicks_diff,
    round(s.spend - m.spend, 2) as spend_diff,
    s.conversions - m.conversions as conversions_diff,
    round(s.revenue - m.revenue, 2) as revenue_diff

from silver s
cross join mart m;

In [0]:
%sql
with base as (
    select *
    from voyage.dbt_dev.fct_campaign_daily 
),                            
campaign_check as (                 
    select                
        count(distinct campaign_id) as campaigns,
        count(distinct tenant_id) as tenants,
        count(*) as rows,             
        min(date) as min_date,                                      
        max(date) as max_date       
    from base   
),              
grain_check as (
    select count(*) as duplicate_groups
    from (
        select
            tenant_id,
            campaign_id,
            date
        from base
        group by tenant_id, campaign_id, date
        having count(*) > 1
    )
),
metric_check as (
    select count(*) as invalid_rows
    from base
    where impressions < 0
       or clicks < 0
       or conversions < 0
       or spend < 0
       or revenue < 0
       or clicks > impressions
       or conversions > clicks
       or ctr < 0
       or ctr > 1
       or cpc < 0
       or conversion_rate < 0
       or conversion_rate > 1
       or roas < 0
),
null_check as (
    select count(*) as rows_with_nulls
    from base
    where campaign_id is null
       or tenant_id is null
       or date is null
       or impressions is null
       or clicks is null
       or spend is null
       or conversions is null
       or revenue is null
       or ctr is null
       or cpc is null
       or conversion_rate is null
       or roas is null
)
select
    case
        when c.campaigns = 6
         and c.tenants = 2
         and c.rows = 1098
         and c.min_date = '2026-04-01'
         and c.max_date = '2026-09-30'
         and g.duplicate_groups = 0
         and m.invalid_rows = 0
         and n.rows_with_nulls = 0
        then 'PASS'
        else 'FAIL'
    end as marketing_domain_status,
    c.campaigns,
    c.tenants,
    c.rows,
    c.min_date,
    c.max_date,
    g.duplicate_groups,
    m.invalid_rows,
    n.rows_with_nulls
from campaign_check c
cross join grain_check g
cross join metric_check m
cross join null_check n;

In [0]:
%sql
select
    channel,
    count(distinct campaign_id) as campaigns,
    count(*) as campaign_days,
    sum(impressions) as impressions,
    sum(clicks) as clicks,
    round(sum(spend), 2) as spend,
    sum(conversions) as conversions,
    round(sum(revenue), 2) as revenue,
    round(sum(revenue) / nullif(sum(spend), 0), 2) as aggregate_roas
from voyage.dbt_dev.fct_campaign_daily
group by channel
order by channel;